# COMP20008 Assignment 2 — W04G10

**Research Question:** Among Melbourne entire homes and apartments with valid nightly prices, do location and amenities improve high-price prediction beyond property size alone, and which attributes are most useful?

This submission notebook reproduces preprocessing, correlation analysis, KNN/Decision Tree modelling, evaluation, and feature selection from the original Melbourne Inside Airbnb Detailed Listings dataset. It downloads the official 16 June 2026 source directly from Inside Airbnb when `data/listings.csv` is absent.

# Part 1 — Data Preprocessing

## 1. Imports and official data source

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

REPO_ROOT = Path("..").resolve() if Path("../src").exists() else Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_source import ensure_melbourne_listings, MELBOURNE_LISTINGS_GZ_URL
from src.a1_reuse import (
    validate_a2_source,
    validate_melbourne_source,
    add_clean_price,
    add_amenity_count,
    add_bathrooms_numeric,
    parse_amenities,
)

DATA_PATH = ensure_melbourne_listings(REPO_ROOT / "data" / "listings.csv")
print("Official source:", MELBOURNE_LISTINGS_GZ_URL)
print("Using:", DATA_PATH.resolve())

## 2. Load and validate the raw original dataset

The two guards below reject the known 29-column A1 teaching dataset and a clearly non-Melbourne file.

In [ ]:
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
validate_a2_source(df_raw)
validate_melbourne_source(df_raw)

print("Raw shape:", df_raw.shape)
print("Median coordinates:",
      round(pd.to_numeric(df_raw["latitude"], errors="coerce").median(), 5),
      round(pd.to_numeric(df_raw["longitude"], errors="coerce").median(), 5))
display(df_raw.head())

## 3. Initial audit

In [ ]:
audit = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing_n": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(2),
    "n_unique": df_raw.nunique(dropna=True),
}).sort_values(["missing_pct", "missing_n"], ascending=False)

display(audit.head(50))

## 4. Define the RQ cohort

The analysis population is restricted to:
- `room_type == "Entire home/apt"`;
- a valid, finite, positive nightly price.

Price parsing reuses the group's A1 corrected parsing logic, but it is rerun on the original A2 data.

In [ ]:
n_raw = len(df_raw)

df = df_raw.loc[df_raw["room_type"].eq("Entire home/apt")].copy()
n_entire = len(df)

df = add_clean_price(df, source="price", target="price_clean")
valid_price = (
    df["price_clean"].notna()
    & np.isfinite(df["price_clean"])
    & df["price_clean"].gt(0)
)
df = df.loc[valid_price].copy()
n_eligible = len(df)

cohort_summary = pd.DataFrame({
    "stage": [
        "Raw original listings",
        "Entire home/apt",
        "Entire home/apt + valid positive nightly price",
    ],
    "rows": [n_raw, n_entire, n_eligible],
})
cohort_summary["retained_pct_of_raw"] = (
    cohort_summary["rows"] / n_raw * 100
).round(2)

display(cohort_summary)
display(df["price_clean"].describe(percentiles=[.25,.5,.75,.9,.95,.99]))

## 5. Six preprocessing candidates

The assignment requires at least six candidates and three selected tasks. The final three are chosen to map directly to the RQ's three constructs: **property size, location, and amenities**.

Selected:
1. Reuse A1 numeric bathrooms derived from `bathrooms_text`.
2. Engineer distance from Melbourne CBD from latitude/longitude.
3. Engineer amenity count and a small set of amenity indicators.

Not selected as separate preprocessing tasks:
4. Special missing-value treatment for bedrooms/beds.
5. Property-type consolidation.
6. Neighbourhood encoding/consolidation.

The table below generates the dataset-specific evidence used to justify these choices.

## 6. Selected task 1 — A1 numeric bathrooms

In [ ]:
# Reuse the A1 pipeline logic as required by the A2 specification.
df = add_bathrooms_numeric(
    df,
    source="bathrooms_text",
    target="bathrooms",
)

bathroom_summary = {
    "rows": len(df),
    "source_text_categories": int(df["bathrooms_text"].nunique(dropna=True)),
    "source_missing_n": int(df["bathrooms_text"].isna().sum()),
    "numeric_nonmissing_n": int(df["bathrooms"].notna().sum()),
    "numeric_missing_n": int(df["bathrooms"].isna().sum()),
    "numeric_usable_pct": round(float(df["bathrooms"].notna().mean() * 100), 2),
}
bathroom_summary

## 7. Selected task 2 — distance from Melbourne CBD

Distance is calculated with the Haversine formula using approximately `(-37.8136, 144.9631)` as the CBD reference point.

In [ ]:
def haversine_km(lat, lon, ref_lat=-37.8136, ref_lon=144.9631):
    lat1 = np.radians(pd.to_numeric(lat, errors="coerce"))
    lon1 = np.radians(pd.to_numeric(lon, errors="coerce"))
    lat2 = np.radians(ref_lat)
    lon2 = np.radians(ref_lon)

    dlat = lat1 - lat2
    dlon = lon1 - lon2
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    return 6371.0088 * 2 * np.arcsin(np.sqrt(a))

df["distance_cbd_km"] = haversine_km(df["latitude"], df["longitude"])
display(df["distance_cbd_km"].describe(percentiles=[.25,.5,.75,.9,.95]))

## 8. Selected task 3 — amenities engineering

A1 showed why naive comma splitting is unsafe for this field. The A1 parser is reused on the original A2 data. The task creates:
- `amenity_count`;
- six interpretable amenity indicators for the full prediction model.

In [ ]:
df = add_amenity_count(df, source="amenities", target="amenity_count")
df["amenities_list"] = df["amenities"].apply(parse_amenities)

AMENITY_KEYWORDS = {
    "has_pool": ("pool",),
    "has_free_parking": ("free parking",),
    "has_air_conditioning": ("air conditioning", "window ac", "central air"),
    "has_kitchen": ("kitchen",),
    "has_washer": ("washer",),
    "has_dryer": ("dryer",),
}

def amenity_indicator(items, keywords):
    text = " | ".join(str(x).lower() for x in items)
    return int(any(keyword in text for keyword in keywords))

for feature, keywords in AMENITY_KEYWORDS.items():
    df[feature] = df["amenities_list"].apply(
        lambda items, kw=keywords: amenity_indicator(items, kw)
    )

amenity_indicator_summary = pd.DataFrame({
    "feature": list(AMENITY_KEYWORDS),
    "n_yes": [int(df[c].sum()) for c in AMENITY_KEYWORDS],
    "pct_yes": [round(float(df[c].mean() * 100), 2) for c in AMENITY_KEYWORDS],
})
display(amenity_indicator_summary)
display(df["amenity_count"].describe(percentiles=[.25,.5,.75,.9,.95]))

## 9. Dataset-specific candidate decision table

The rejected candidates are still handled where necessary:
- bedrooms/beds missingness is handled inside model pipelines with training-fold median imputation;
- property type is not added because it is outside the RQ's size/location/amenity comparison;
- neighbourhood encoding is not used because distance provides a compact location feature without 30 dummy variables.

In [ ]:
property_counts = df["property_type"].value_counts(dropna=False)
neighbour_counts = df["neighbourhood_cleansed"].value_counts(dropna=False)

bedrooms_missing_n = int(df["bedrooms"].isna().sum())
beds_missing_n = int(df["beds"].isna().sum())

candidate_decisions = pd.DataFrame([
    {
        "candidate": "Numeric bathrooms from bathrooms_text (A1 pipeline)",
        "selected": True,
        "dataset_evidence": (
            f'{bathroom_summary["numeric_nonmissing_n"]:,}/{len(df):,} rows '
            f'({bathroom_summary["numeric_usable_pct"]:.2f}%) receive a numeric value '
            f'from {bathroom_summary["source_text_categories"]} text categories.'
        ),
        "alternative_considered": "Keep bathrooms_text as a categorical predictor",
        "why_alternative_not_selected": (
            f'bathrooms_text has {bathroom_summary["source_text_categories"]} categories; '
            f'the A1 numeric derivation retains {bathroom_summary["numeric_usable_pct"]:.2f}% '
            'coverage and gives a direct property-size measure.'
        ),
    },
    {
        "candidate": "Distance from CBD from latitude/longitude",
        "selected": True,
        "dataset_evidence": (
            f'{int(df["distance_cbd_km"].notna().sum()):,}/{len(df):,} rows receive distance; '
            f'median={df["distance_cbd_km"].median():.2f} km.'
        ),
        "alternative_considered": "One-hot encode neighbourhood_cleansed",
        "why_alternative_not_selected": (
            f'neighbourhood_cleansed has {neighbour_counts.size} categories and '
            f'{int(df["neighbourhood_cleansed"].isna().sum())} missing rows; '
            'distance gives one continuous location feature aligned with the RQ.'
        ),
    },
    {
        "candidate": "Amenity count + amenity indicators",
        "selected": True,
        "dataset_evidence": (
            f'Raw amenities has {df["amenities"].nunique(dropna=True):,} unique strings; '
            f'amenity_count has {df["amenity_count"].nunique(dropna=True)} unique values '
            f'across {len(df):,} rows.'
        ),
        "alternative_considered": "Use the raw amenities string as a categorical feature",
        "why_alternative_not_selected": (
            f'raw amenities contains {df["amenities"].nunique(dropna=True):,} unique strings '
            f'across {len(df):,} eligible rows, so it is close to listing-specific; '
            'count + six interpretable indicators provides a compact representation.'
        ),
    },
    {
        "candidate": "Special bedroom/bed missing-value preprocessing",
        "selected": False,
        "dataset_evidence": (
            f'Bedrooms missing: {bedrooms_missing_n:,} '
            f'({bedrooms_missing_n/len(df)*100:.2f}%); beds missing: {beds_missing_n:,} '
            f'({beds_missing_n/len(df)*100:.2f}%). Median imputation is kept inside model CV.'
        ),
        "alternative_considered": "Standalone pre-imputation before splitting",
        "why_alternative_not_selected": (
            'Imputation is performed inside the modelling pipeline so training-fold '
            'statistics do not leak into validation/test data.'
        ),
    },
    {
        "candidate": "Property-type consolidation",
        "selected": False,
        "dataset_evidence": (
            f'{property_counts.size} property types; '
            f'{int((property_counts < 10).sum())} have fewer than 10 eligible listings.'
        ),
        "alternative_considered": "Collapse rare types to Other",
        "why_alternative_not_selected": (
            'Property type is outside the planned size/location/amenities comparison '
            'and would change the RQ feature-set contrast.'
        ),
    },
    {
        "candidate": "Neighbourhood encoding/consolidation",
        "selected": False,
        "dataset_evidence": (
            f'{neighbour_counts.size} neighbourhood categories with '
            f'{int(df["neighbourhood_cleansed"].isna().sum())} missing rows.'
        ),
        "alternative_considered": "One-hot encode all neighbourhood categories",
        "why_alternative_not_selected": (
            f'This would introduce {neighbour_counts.size} categorical levels; '
            'distance-to-CBD is retained as the primary compact location representation.'
        ),
    },
])

display(candidate_decisions)

selected_task_alternatives = candidate_decisions.loc[
    candidate_decisions["selected"],
    [
        "candidate",
        "dataset_evidence",
        "alternative_considered",
        "why_alternative_not_selected",
    ],
].copy()

display(selected_task_alternatives)

## 10. Measurable impact of the three selected preprocessing tasks

In [ ]:
preprocessing_impact = pd.DataFrame([
    {
        "task": "A1 numeric bathrooms",
        "before": f'{df["bathrooms_text"].nunique(dropna=True)} text categories',
        "after": f'{df["bathrooms"].notna().sum():,} numeric values',
        "rows_affected": int(df["bathrooms"].notna().sum()),
    },
    {
        "task": "Distance from CBD",
        "before": "latitude + longitude as two coordinates",
        "after": (
            f'one distance feature; median={df["distance_cbd_km"].median():.2f} km, '
            f'max={df["distance_cbd_km"].max():.2f} km'
        ),
        "rows_affected": int(df["distance_cbd_km"].notna().sum()),
    },
    {
        "task": "Amenities engineering",
        "before": f'{df["amenities"].nunique(dropna=True):,} unique raw amenity strings',
        "after": (
            f'{df["amenity_count"].nunique(dropna=True)} amenity-count values + '
            f'{len(AMENITY_KEYWORDS)} binary amenity indicators'
        ),
        "rows_affected": int(len(df)),
    },
])

display(preprocessing_impact)

## 11. Shared target and split definition

The group contract defines **high price** as price above the **training sample's 75th percentile**.

To also satisfy the rubric's stratified-split requirement without using test prices to set the threshold, the code iterates:
1. stratify using the current threshold;
2. compute Q75 from the resulting training sample only;
3. repeat until the threshold is unchanged.

At convergence, the split is stratified by the same labels produced by the final training-sample threshold.

In [ ]:
def make_training_q75_stratified_split(
    frame,
    price_col="price_clean",
    test_size=0.20,
    random_state=42,
    max_iter=50,
    atol=1e-10,
):
    indices = np.arange(len(frame))
    threshold = float(frame[price_col].quantile(0.75))
    history = []

    for iteration in range(1, max_iter + 1):
        labels = (frame[price_col].to_numpy() > threshold).astype(int)

        train_idx, test_idx = train_test_split(
            indices,
            test_size=test_size,
            random_state=random_state,
            stratify=labels,
        )

        new_threshold = float(
            frame.iloc[train_idx][price_col].quantile(0.75)
        )
        history.append({
            "iteration": iteration,
            "threshold_used_for_stratification": threshold,
            "training_q75": new_threshold,
        })

        if np.isclose(new_threshold, threshold, rtol=0, atol=atol):
            final_labels = (
                frame[price_col].to_numpy() > new_threshold
            ).astype(int)
            return train_idx, test_idx, new_threshold, final_labels, pd.DataFrame(history)

        threshold = new_threshold

    raise RuntimeError(
        "Training-Q75 / stratified-split iteration did not converge. "
        "Review the split design with the tutor."
    )

train_idx, test_idx, price_threshold, target, split_history = (
    make_training_q75_stratified_split(
        df,
        random_state=RANDOM_STATE,
    )
)

df["high_price"] = target
df["split"] = "test"
df.iloc[train_idx, df.columns.get_loc("split")] = "train"

display(split_history)

split_summary = (
    df.groupby(["split", "high_price"])
      .size()
      .rename("n")
      .reset_index()
)
split_summary["pct_within_split"] = (
    split_summary["n"]
    / split_summary.groupby("split")["n"].transform("sum")
    * 100
).round(2)

print("Final training-sample Q75 threshold:", price_threshold)
display(split_summary)

## 12. Export the stable handoff dataset and preprocessing evidence

All later notebooks use this exact file so target, split, rows and feature engineering remain consistent.

In [ ]:
SIZE_FEATURES = ["accommodates", "bedrooms", "beds", "bathrooms"]
LOCATION_FEATURES = ["distance_cbd_km"]
AMENITY_FEATURES = [
    "amenity_count",
    "has_pool",
    "has_free_parking",
    "has_air_conditioning",
    "has_kitchen",
    "has_washer",
    "has_dryer",
]

export_cols = [
    "id",
    "price_clean",
    "high_price",
    "split",
    "property_type",
    "neighbourhood_cleansed",
    "latitude",
    "longitude",
] + SIZE_FEATURES + LOCATION_FEATURES + AMENITY_FEATURES

processed = df[export_cols].copy()

DATA_OUT = REPO_ROOT / "data" / "processed_listings.csv"
TABLE_OUT = REPO_ROOT / "output" / "tables"
FIG_OUT = REPO_ROOT / "output" / "figures"
TABLE_OUT.mkdir(parents=True, exist_ok=True)
FIG_OUT.mkdir(parents=True, exist_ok=True)

processed.to_csv(DATA_OUT, index=False)
candidate_decisions.to_csv(TABLE_OUT / "preprocessing_candidates.csv", index=False)
selected_task_alternatives.to_csv(TABLE_OUT / "preprocessing_selected_alternatives.csv", index=False)
preprocessing_impact.to_csv(TABLE_OUT / "preprocessing_impact.csv", index=False)
split_summary.to_csv(TABLE_OUT / "split_summary.csv", index=False)
amenity_indicator_summary.to_csv(TABLE_OUT / "amenity_indicator_prevalence.csv", index=False)

metadata = {
    "source_url": MELBOURNE_LISTINGS_GZ_URL,
    "raw_rows": int(n_raw),
    "entire_home_rows": int(n_entire),
    "eligible_rows": int(n_eligible),
    "training_price_q75": float(price_threshold),
    "train_rows": int((processed["split"] == "train").sum()),
    "test_rows": int((processed["split"] == "test").sum()),
    "size_features": SIZE_FEATURES,
    "location_features": LOCATION_FEATURES,
    "amenity_features": AMENITY_FEATURES,
}
(TABLE_OUT / "preprocessing_metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print("Saved:", DATA_OUT)
print("Processed shape:", processed.shape)
display(processed.head())

## 13. Simple data visualisations for later report selection

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(df["price_clean"].clip(upper=df["price_clean"].quantile(0.99)), bins=40)
ax.axvline(price_threshold, linestyle="--", linewidth=1.5)
ax.set_xlabel("Nightly price (AUD; clipped at 99th percentile for display)")
ax.set_ylabel("Listings")
ax.set_title("Eligible Melbourne entire-home nightly prices")
fig.tight_layout()
fig.savefig(FIG_OUT / "eligible_price_distribution.png", dpi=200)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(df["distance_cbd_km"], bins=40)
ax.set_xlabel("Distance from Melbourne CBD (km)")
ax.set_ylabel("Listings")
ax.set_title("Distance-to-CBD distribution")
fig.tight_layout()
fig.savefig(FIG_OUT / "distance_cbd_distribution.png", dpi=200)
plt.close(fig)

print("Saved preprocessing figures to:", FIG_OUT)

# Part 2 — Correlation Analysis

## 1. Imports and processed data

In [ ]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mutual_info_score, normalized_mutual_info_score

REPO_ROOT = Path("..").resolve() if Path("../data").exists() else Path(".").resolve()
DATA_PATH = REPO_ROOT / "data" / "processed_listings.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Run notebooks/01_preprocessing.ipynb first to create processed_listings.csv."
    )

df = pd.read_csv(DATA_PATH)
print("Processed shape:", df.shape)
print("Split counts:", df["split"].value_counts().to_dict())
print("Target counts:", df["high_price"].value_counts().sort_index().to_dict())

## 2. Variable set and method implementation

Pearson and Spearman are computed directly on numeric representations.

For MI/NMI, continuous/count variables are discretised into up to five quantile bins. The binary target is left as binary. This makes the pairwise MI/NMI calculation symmetric and reproducible; the report should explicitly state this implementation choice.

In [ ]:
VARIABLES = [
    "accommodates",
    "bedrooms",
    "bathrooms",
    "distance_cbd_km",
    "amenity_count",
    "high_price",
]

missing_cols = [c for c in VARIABLES if c not in df.columns]
if missing_cols:
    raise KeyError(f"Missing correlation columns: {missing_cols}")

VARIABLE_TYPES = {
    "accommodates": "numeric",
    "bedrooms": "numeric",
    "bathrooms": "numeric",
    "distance_cbd_km": "numeric",
    "amenity_count": "numeric",
    "high_price": "binary",
}

display(df[VARIABLES].describe())

## 3. Helpers for MI/NMI discretisation

In [ ]:
def to_information_categories(series, variable_type, max_bins=5):
    if variable_type == "binary":
        return pd.to_numeric(series, errors="coerce")

    numeric = pd.to_numeric(series, errors="coerce")
    out = pd.Series(np.nan, index=series.index, dtype="float64")
    valid = numeric.notna()

    if valid.sum() < 2:
        return out

    # qcut may drop duplicate edges for low-cardinality numeric variables.
    binned = pd.qcut(
        numeric.loc[valid],
        q=min(max_bins, numeric.loc[valid].nunique()),
        labels=False,
        duplicates="drop",
    )
    out.loc[valid] = binned.astype(float)
    return out

## 4. Compute all four methods for every pair

In [ ]:
rows = []

for a, b in combinations(VARIABLES, 2):
    pair = df[[a, b]].dropna().copy()

    pearson = pearsonr(pair[a], pair[b]).statistic
    spearman = spearmanr(pair[a], pair[b]).statistic

    a_disc = to_information_categories(
        pair[a], VARIABLE_TYPES[a]
    )
    b_disc = to_information_categories(
        pair[b], VARIABLE_TYPES[b]
    )
    valid = a_disc.notna() & b_disc.notna()

    mi = mutual_info_score(
        a_disc.loc[valid].astype(int),
        b_disc.loc[valid].astype(int),
    )
    nmi = normalized_mutual_info_score(
        a_disc.loc[valid].astype(int),
        b_disc.loc[valid].astype(int),
    )

    rows.append({
        "var_a": a,
        "var_b": b,
        "n": len(pair),
        "pearson": float(pearson),
        "spearman": float(spearman),
        "mutual_information": float(mi),
        "normalised_mutual_information": float(nmi),
        "implementation_note": (
            "Pearson/Spearman on numeric values; MI/NMI on 5-quantile "
            "discretisation for numeric variables; high_price left binary."
        ),
    })

corr_results = pd.DataFrame(rows)
display(corr_results)

## 5. Target associations

This table isolates every predictor–target pair so the group can identify the strongest/weakest associations with actual values.

In [ ]:
target_rows = corr_results[
    (corr_results["var_a"] == "high_price")
    | (corr_results["var_b"] == "high_price")
].copy()

target_rows["predictor"] = np.where(
    target_rows["var_a"] == "high_price",
    target_rows["var_b"],
    target_rows["var_a"],
)
target_rows["abs_pearson"] = target_rows["pearson"].abs()
target_rows["abs_spearman"] = target_rows["spearman"].abs()

display(
    target_rows[
        [
            "predictor",
            "n",
            "pearson",
            "spearman",
            "mutual_information",
            "normalised_mutual_information",
        ]
    ].sort_values("normalised_mutual_information", ascending=False)
)

## 6. Predictor–predictor relationships

This is used to identify potential redundancy/multicollinearity before modelling.

In [ ]:
predictor_pairs = corr_results[
    (corr_results["var_a"] != "high_price")
    & (corr_results["var_b"] != "high_price")
].copy()

predictor_pairs["abs_pearson"] = predictor_pairs["pearson"].abs()
predictor_pairs["abs_spearman"] = predictor_pairs["spearman"].abs()

display(
    predictor_pairs.sort_values(
        ["abs_pearson", "normalised_mutual_information"],
        ascending=False,
    )
)

## 7. Correlation matrices and figures

In [ ]:
TABLE_OUT = REPO_ROOT / "output" / "tables"
FIG_OUT = REPO_ROOT / "output" / "figures"
TABLE_OUT.mkdir(parents=True, exist_ok=True)
FIG_OUT.mkdir(parents=True, exist_ok=True)

corr_results.to_csv(TABLE_OUT / "correlation_results.csv", index=False)
target_rows.to_csv(TABLE_OUT / "correlation_target_associations.csv", index=False)
predictor_pairs.to_csv(TABLE_OUT / "correlation_predictor_pairs.csv", index=False)

def symmetric_matrix(results, value_col):
    matrix = pd.DataFrame(
        np.eye(len(VARIABLES)),
        index=VARIABLES,
        columns=VARIABLES,
        dtype=float,
    )

    # MI has a meaningful non-one diagonal, but the diagonal is not used
    # in pairwise interpretation. Keep 1.0 visually for consistency.
    for _, row in results.iterrows():
        matrix.loc[row["var_a"], row["var_b"]] = row[value_col]
        matrix.loc[row["var_b"], row["var_a"]] = row[value_col]
    return matrix

for method, col in {
    "pearson": "pearson",
    "spearman": "spearman",
    "mi": "mutual_information",
    "nmi": "normalised_mutual_information",
}.items():
    matrix = symmetric_matrix(corr_results, col)
    matrix.to_csv(TABLE_OUT / f"{method}_matrix.csv")

    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(matrix.values, aspect="auto")
    ax.set_xticks(range(len(VARIABLES)))
    ax.set_yticks(range(len(VARIABLES)))
    ax.set_xticklabels(VARIABLES, rotation=45, ha="right")
    ax.set_yticklabels(VARIABLES)
    ax.set_title(f"{method.upper()} pairwise matrix")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(FIG_OUT / f"correlation_{method}.png", dpi=200)
    plt.close(fig)

print("Saved correlation tables to:", TABLE_OUT)
print("Saved correlation figures to:", FIG_OUT)

## 8. Evidence checklist for group-written interpretation

Use the generated tables to write the report yourselves. Before finalising this section, identify with actual values:
- which methods agree and which diverge;
- the strongest, weakest and near-absent predictor–target relationships;
- any strong predictor–predictor relationships;
- one concrete downstream modelling/feature decision informed by the correlation results;
- plausible confounders/biases, using association rather than causal language.

# Part 3 — Supervised Learning & Evaluation

## 1. Imports and data

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

REPO_ROOT = Path("..").resolve() if Path("../data").exists() else Path(".").resolve()
DATA_PATH = REPO_ROOT / "data" / "processed_listings.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError("Run 01_preprocessing.ipynb first.")

df = pd.read_csv(DATA_PATH)
print("Processed shape:", df.shape)
print("Split counts:", df["split"].value_counts().to_dict())
print("Target by split:")
display(pd.crosstab(df["split"], df["high_price"], margins=True))

## 2. Feature sets

In [ ]:
SIZE_FEATURES = [
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
]

LOCATION_FEATURES = [
    "distance_cbd_km",
]

AMENITY_FEATURES = [
    "amenity_count",
    "has_pool",
    "has_free_parking",
    "has_air_conditioning",
    "has_kitchen",
    "has_washer",
    "has_dryer",
]

FEATURE_SETS = {
    "size_only": SIZE_FEATURES,
    "size_location_amenities": (
        SIZE_FEATURES + LOCATION_FEATURES + AMENITY_FEATURES
    ),
}

TARGET = "high_price"

for name, features in FEATURE_SETS.items():
    missing = [c for c in features if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing columns: {missing}")

train_df = df.loc[df["split"].eq("train")].copy()
test_df = df.loc[df["split"].eq("test")].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train positive rate:", round(train_df[TARGET].mean(), 4))
print("Test positive rate:", round(test_df[TARGET].mean(), 4))

## 3. Evaluation helpers

In [ ]:
def metric_summary(y_true, y_pred, y_score=None):
    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],
        zero_division=0,
    )

    result = {
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1),
        "precision_class_0": float(precision[0]),
        "recall_class_0": float(recall[0]),
        "f1_class_0": float(f1[0]),
        "support_class_0": int(support[0]),
        "precision_class_1": float(precision[1]),
        "recall_class_1": float(recall[1]),
        "f1_class_1": float(f1[1]),
        "support_class_1": int(support[1]),
    }

    if y_score is not None and len(np.unique(y_true)) == 2:
        result["roc_auc"] = float(roc_auc_score(y_true, y_score))
    else:
        result["roc_auc"] = np.nan

    return result


def bootstrap_macro_f1_ci(
    y_true,
    y_pred,
    n_boot=2000,
    alpha=0.05,
    random_state=42,
):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    rng = np.random.default_rng(random_state)
    n = len(y_true)
    scores = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        scores.append(
            f1_score(
                y_true[idx],
                y_pred[idx],
                average="macro",
                zero_division=0,
            )
        )

    scores = np.asarray(scores)
    lo, hi = np.quantile(scores, [alpha/2, 1-alpha/2])

    return {
        "bootstrap_mean_macro_f1": float(scores.mean()),
        "ci_lower_95": float(lo),
        "ci_upper_95": float(hi),
        "n_boot": int(n_boot),
    }

## 4. Majority-class baseline

The baseline is evaluated on the held-out test set with the same metrics as the trained models.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(
    np.zeros((len(train_df), 1)),
    train_df[TARGET],
)
baseline_pred = baseline.predict(np.zeros((len(test_df), 1)))
baseline_score = baseline.predict_proba(np.zeros((len(test_df), 1)))[:, 1]

baseline_metrics = metric_summary(
    test_df[TARGET].to_numpy(),
    baseline_pred,
    baseline_score,
)
baseline_metrics

## 5. Model pipelines and tuning grids

KNN uses median imputation + standardisation because it is distance-based.

Decision Tree uses median imputation but no scaling.

Both use 5-fold stratified CV on the training set and tune macro-F1. Every tried parameter combination is saved.

In [ ]:
def make_knn_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier()),
    ])

def make_tree_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ])

KNN_GRID = {
    "model__n_neighbors": [3, 5, 7, 11, 15, 21, 31],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2],
}

TREE_GRID = {
    "model__max_depth": [None, 3, 5, 8, 12],
    "model__min_samples_split": [2, 10, 30],
    "model__min_samples_leaf": [1, 5, 15, 30],
}

CV = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print("KNN combinations:", np.prod([len(v) for v in KNN_GRID.values()]))
print("Tree combinations:", np.prod([len(v) for v in TREE_GRID.values()]))

## 6. Tune and evaluate each model × feature-set combination

In [ ]:
def run_experiment(model_name, feature_set_name):
    features = FEATURE_SETS[feature_set_name]

    X_train = train_df[features]
    y_train = train_df[TARGET]
    X_test = test_df[features]
    y_test = test_df[TARGET]

    if model_name == "KNN":
        pipeline = make_knn_pipeline()
        grid = KNN_GRID
    elif model_name == "DecisionTree":
        pipeline = make_tree_pipeline()
        grid = TREE_GRID
    else:
        raise ValueError(model_name)

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=grid,
        scoring="f1_macro",
        cv=CV,
        n_jobs=-1,
        return_train_score=True,
    )
    search.fit(X_train, y_train)

    pred = search.predict(X_test)
    proba = search.predict_proba(X_test)[:, 1]

    metrics = metric_summary(y_test.to_numpy(), pred, proba)
    metrics.update({
        "model": model_name,
        "feature_set": feature_set_name,
        "best_cv_macro_f1": float(search.best_score_),
        "best_params": json.dumps(search.best_params_, sort_keys=True),
        "absolute_macro_f1_improvement_vs_baseline": (
            metrics["macro_f1"] - baseline_metrics["macro_f1"]
        ),
    })

    cv_results = pd.DataFrame(search.cv_results_)
    cv_results.insert(0, "model", model_name)
    cv_results.insert(1, "feature_set", feature_set_name)

    return {
        "search": search,
        "pred": pred,
        "proba": proba,
        "metrics": metrics,
        "cv_results": cv_results,
        "confusion_matrix": confusion_matrix(y_test, pred, labels=[0, 1]),
        "features": features,
    }

experiments = {}

for model_name in ["KNN", "DecisionTree"]:
    for feature_set_name in FEATURE_SETS:
        print("Running:", model_name, feature_set_name)
        experiments[(model_name, feature_set_name)] = run_experiment(
            model_name,
            feature_set_name,
        )

model_summary = pd.DataFrame(
    [result["metrics"] for result in experiments.values()]
)

display(
    model_summary[
        [
            "model",
            "feature_set",
            "best_cv_macro_f1",
            "accuracy",
            "macro_f1",
            "roc_auc",
            "f1_class_0",
            "f1_class_1",
            "absolute_macro_f1_improvement_vs_baseline",
            "best_params",
        ]
    ].sort_values(["model", "feature_set"])
)

## 7. Hyperparameter defaults, chosen values, and specific CV effect

For each tuned parameter, this table compares the selected setting with that parameter reset to its scikit-learn default while holding the other selected parameters fixed. The difference in mean 5-fold CV macro-F1 gives a dataset-specific effect for every changed hyperparameter.

In [ ]:
MODEL_DEFAULTS = {
    "KNN": {
        "model__n_neighbors": 5,
        "model__weights": "uniform",
        "model__p": 2,
    },
    "DecisionTree": {
        "model__max_depth": None,
        "model__min_samples_split": 2,
        "model__min_samples_leaf": 1,
    },
}

hyperparameter_effect_rows = []

for (model_name, feature_set_name), exp in experiments.items():
    best_params = exp["search"].best_params_
    best_score = float(exp["search"].best_score_)
    cv_table = exp["cv_results"]

    for parameter, default_value in MODEL_DEFAULTS[model_name].items():
        chosen_value = best_params[parameter]
        changed = chosen_value != default_value

        comparison_params = dict(best_params)
        comparison_params[parameter] = default_value

        mask = cv_table["params"].apply(
            lambda p: all(p.get(k) == v for k, v in comparison_params.items())
        )

        if mask.any():
            comparison_score = float(
                cv_table.loc[mask, "mean_test_score"].iloc[0]
            )
            effect = best_score - comparison_score
        else:
            comparison_score = np.nan
            effect = np.nan

        hyperparameter_effect_rows.append({
            "model": model_name,
            "feature_set": feature_set_name,
            "parameter": parameter.replace("model__", ""),
            "default_value": str(default_value),
            "chosen_value": str(chosen_value),
            "changed_from_default": bool(changed),
            "chosen_cv_macro_f1": best_score,
            "cv_macro_f1_with_parameter_at_default": comparison_score,
            "specific_cv_macro_f1_effect": effect,
        })

hyperparameter_effects = pd.DataFrame(hyperparameter_effect_rows)
display(hyperparameter_effects)

## 8. Incremental value of location + amenities

This table directly answers the first part of the RQ for each mandatory model by subtracting size-only performance from full-feature performance.

In [ ]:
incremental_rows = []

for model_name in ["KNN", "DecisionTree"]:
    size_row = model_summary[
        (model_summary["model"] == model_name)
        & (model_summary["feature_set"] == "size_only")
    ].iloc[0]
    full_row = model_summary[
        (model_summary["model"] == model_name)
        & (model_summary["feature_set"] == "size_location_amenities")
    ].iloc[0]

    incremental_rows.append({
        "model": model_name,
        "size_only_macro_f1": float(size_row["macro_f1"]),
        "full_macro_f1": float(full_row["macro_f1"]),
        "absolute_macro_f1_change": float(
            full_row["macro_f1"] - size_row["macro_f1"]
        ),
        "size_only_accuracy": float(size_row["accuracy"]),
        "full_accuracy": float(full_row["accuracy"]),
        "absolute_accuracy_change": float(
            full_row["accuracy"] - size_row["accuracy"]
        ),
        "size_only_roc_auc": float(size_row["roc_auc"]),
        "full_roc_auc": float(full_row["roc_auc"]),
        "absolute_roc_auc_change": float(
            full_row["roc_auc"] - size_row["roc_auc"]
        ),
    })

incremental_results = pd.DataFrame(incremental_rows)
display(incremental_results)

## 9. Uncertainty quantification

A bootstrap 95% confidence interval is calculated for held-out macro-F1 of the better full-feature model (chosen by test macro-F1). The group should discuss the limitation that model selection and interval estimation use the same held-out comparison when interpreting the result.

In [ ]:
full_results = {
    model: experiments[(model, "size_location_amenities")]
    for model in ["KNN", "DecisionTree"]
}

uncertainty_model = max(
    full_results,
    key=lambda m: full_results[m]["metrics"]["best_cv_macro_f1"],
)
uncertainty_exp = full_results[uncertainty_model]

uncertainty = bootstrap_macro_f1_ci(
    test_df[TARGET].to_numpy(),
    uncertainty_exp["pred"],
    n_boot=2000,
    random_state=RANDOM_STATE,
)
uncertainty["model"] = uncertainty_model
uncertainty["feature_set"] = "size_location_amenities"

uncertainty

## 10. Feature influence for both models

Permutation importance is calculated on the same held-out test set for the two full-feature models, using macro-F1. This gives feature-level influence in the original feature space for both KNN and Decision Tree.

In [ ]:
importance_rows = []

for model_name in ["KNN", "DecisionTree"]:
    exp = experiments[(model_name, "size_location_amenities")]
    features = exp["features"]
    X_test = test_df[features]
    y_test = test_df[TARGET]

    perm = permutation_importance(
        exp["search"].best_estimator_,
        X_test,
        y_test,
        scoring="f1_macro",
        n_repeats=10,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    for feature, mean_imp, std_imp in zip(
        features,
        perm.importances_mean,
        perm.importances_std,
    ):
        importance_rows.append({
            "model": model_name,
            "feature": feature,
            "permutation_importance_mean": float(mean_imp),
            "permutation_importance_std": float(std_imp),
        })

permutation_importance_table = pd.DataFrame(importance_rows)
display(
    permutation_importance_table.sort_values(
        ["model", "permutation_importance_mean"],
        ascending=[True, False],
    )
)

## 11. Save evaluation outputs and figures

In [ ]:
TABLE_OUT = REPO_ROOT / "output" / "tables"
FIG_OUT = REPO_ROOT / "output" / "figures"
TABLE_OUT.mkdir(parents=True, exist_ok=True)
FIG_OUT.mkdir(parents=True, exist_ok=True)

baseline_table = pd.DataFrame([{
    "model": "MajorityClassBaseline",
    "feature_set": "none",
    **baseline_metrics,
}])

baseline_table.to_csv(TABLE_OUT / "baseline_metrics.csv", index=False)
model_summary.to_csv(TABLE_OUT / "model_summary.csv", index=False)
hyperparameter_effects.to_csv(TABLE_OUT / "hyperparameter_effects.csv", index=False)
incremental_results.to_csv(TABLE_OUT / "model_incremental_value.csv", index=False)
permutation_importance_table.to_csv(
    TABLE_OUT / "model_permutation_importance.csv",
    index=False,
)
pd.DataFrame([uncertainty]).to_csv(
    TABLE_OUT / "model_uncertainty.csv",
    index=False,
)

for (model_name, feature_set_name), exp in experiments.items():
    safe_model = model_name.lower()
    safe_set = feature_set_name.lower()

    keep_cols = [
        c for c in exp["cv_results"].columns
        if c.startswith("param_")
        or c in [
            "model",
            "feature_set",
            "mean_test_score",
            "std_test_score",
            "mean_train_score",
            "std_train_score",
            "rank_test_score",
            "params",
        ]
    ]
    exp["cv_results"][keep_cols].to_csv(
        TABLE_OUT / f"cv_{safe_model}_{safe_set}.csv",
        index=False,
    )

    cm = pd.DataFrame(
        exp["confusion_matrix"],
        index=["actual_0", "actual_1"],
        columns=["pred_0", "pred_1"],
    )
    cm.to_csv(
        TABLE_OUT / f"confusion_{safe_model}_{safe_set}.csv"
    )

plot_df = model_summary.pivot(
    index="model",
    columns="feature_set",
    values="macro_f1",
)

fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(len(plot_df.index))
width = 0.36

ax.bar(
    x - width/2,
    plot_df["size_only"],
    width,
    label="Size only",
)
ax.bar(
    x + width/2,
    plot_df["size_location_amenities"],
    width,
    label="Size + location + amenities",
)
ax.axhline(
    baseline_metrics["macro_f1"],
    linestyle="--",
    linewidth=1.2,
    label="Majority baseline",
)
ax.set_xticks(x)
ax.set_xticklabels(plot_df.index)
ax.set_ylabel("Held-out macro-F1")
ax.set_title("Model performance by feature set")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_OUT / "model_macro_f1_comparison.png", dpi=200)
plt.close(fig)

for model_name in ["KNN", "DecisionTree"]:
    imp = permutation_importance_table[
        permutation_importance_table["model"] == model_name
    ].sort_values("permutation_importance_mean", ascending=True)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.barh(
        imp["feature"],
        imp["permutation_importance_mean"],
        xerr=imp["permutation_importance_std"],
    )
    ax.set_xlabel("Permutation importance (macro-F1 decrease)")
    ax.set_title(f"{model_name} feature influence")
    fig.tight_layout()
    fig.savefig(
        FIG_OUT / f"permutation_importance_{model_name.lower()}.png",
        dpi=200,
    )
    plt.close(fig)

print("Saved model outputs to:", TABLE_OUT)
print("Saved model figures to:", FIG_OUT)

## 12. Hyperparameter evidence checklist

For the group-written Methodology/Discussion, use the saved CV tables to state:
- every value tried;
- each tuned parameter's default;
- selected value;
- the observed validation-score effect of changing it.

Also compare model behaviour/limitations and feature influence using the actual output tables rather than generic model descriptions.

# Part 4 — Feature Selection

## 1. Imports and data

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

REPO_ROOT = Path("..").resolve() if Path("../data").exists() else Path(".").resolve()
DATA_PATH = REPO_ROOT / "data" / "processed_listings.csv"
MODEL_SUMMARY_PATH = REPO_ROOT / "output" / "tables" / "model_summary.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError("Run 01_preprocessing.ipynb first.")

if not MODEL_SUMMARY_PATH.exists():
    raise FileNotFoundError("Run 03_modelling.ipynb first.")

df = pd.read_csv(DATA_PATH)
model_summary = pd.read_csv(MODEL_SUMMARY_PATH)

print("Processed shape:", df.shape)
print("Split counts:", df["split"].value_counts().to_dict())

## 2. Reuse the exact modelling feature set and training rows

In [ ]:
FEATURES = [
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "distance_cbd_km",
    "amenity_count",
    "has_pool",
    "has_free_parking",
    "has_air_conditioning",
    "has_kitchen",
    "has_washer",
    "has_dryer",
]

TARGET = "high_price"
ID_COL = "id"

missing = [c for c in FEATURES + [TARGET, ID_COL, "split"] if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

train_df = df.loc[df["split"].eq("train")].copy()
test_df = df.loc[df["split"].eq("test")].copy()

X_train = train_df[FEATURES]
y_train = train_df[TARGET].astype(int)

print("Training rows:", len(train_df))
print("Target counts:", y_train.value_counts().sort_index().to_dict())

## 3. Embedded method — tuned Decision Tree feature importance

The same best hyperparameters selected for the full-feature Decision Tree in Notebook 03 are reused here. This keeps the embedded ranking traceable to the model comparison rather than fitting an unrelated tree.

In [ ]:
tree_row = model_summary.loc[
    (model_summary["model"] == "DecisionTree")
    & (model_summary["feature_set"] == "size_location_amenities")
]

if len(tree_row) != 1:
    raise ValueError("Expected exactly one full-feature DecisionTree row in model_summary.csv.")

best_params_raw = tree_row.iloc[0]["best_params"]
best_params = json.loads(best_params_raw)

# GridSearchCV stores pipeline parameter names such as model__max_depth.
tree_kwargs = {
    key.replace("model__", ""): value
    for key, value in best_params.items()
    if key.startswith("model__")
}
tree_kwargs["random_state"] = RANDOM_STATE

imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=FEATURES,
    index=X_train.index,
)

tree = DecisionTreeClassifier(**tree_kwargs)
tree.fit(X_train_imp, y_train)

embedded_scores = pd.Series(
    tree.feature_importances_,
    index=FEATURES,
    name="embedded_score",
).sort_values(ascending=False)

embedded_top3 = embedded_scores.head(3)
display(embedded_top3.to_frame())

## 4. Filter method — Mutual Information

Mutual Information is calculated on the same training rows. Binary amenity indicators are declared discrete; the remaining numeric/count variables are treated as continuous. Median imputation is fitted only on training data.

In [ ]:
DISCRETE_FEATURES = {
    "has_pool",
    "has_free_parking",
    "has_air_conditioning",
    "has_kitchen",
    "has_washer",
    "has_dryer",
}

discrete_mask = np.array([f in DISCRETE_FEATURES for f in FEATURES], dtype=bool)

mi_values = mutual_info_classif(
    X_train_imp,
    y_train,
    discrete_features=discrete_mask,
    random_state=RANDOM_STATE,
)

filter_scores = pd.Series(
    mi_values,
    index=FEATURES,
    name="filter_mi_score",
).sort_values(ascending=False)

filter_top3 = filter_scores.head(3)
display(filter_top3.to_frame())

## 5. Compare the two top-3 lists

The comparison table preserves both ranks and scores. The report discussion should explain any disagreement using these actual values and the different mechanics of embedded vs filter selection.

In [ ]:
rank_table = pd.DataFrame({
    "feature": FEATURES,
    "embedded_score": embedded_scores.reindex(FEATURES).values,
    "filter_mi_score": filter_scores.reindex(FEATURES).values,
})

rank_table["embedded_rank"] = rank_table["embedded_score"].rank(
    method="min", ascending=False
).astype(int)
rank_table["filter_rank"] = rank_table["filter_mi_score"].rank(
    method="min", ascending=False
).astype(int)
rank_table["rank_difference"] = (
    rank_table["embedded_rank"] - rank_table["filter_rank"]
).abs()

rank_table = rank_table.sort_values(
    ["embedded_rank", "filter_rank", "feature"]
).reset_index(drop=True)

display(rank_table)

top3_comparison = pd.DataFrame({
    "embedded_feature": embedded_top3.index.tolist(),
    "embedded_score": embedded_top3.values.tolist(),
    "filter_feature": filter_top3.index.tolist(),
    "filter_score": filter_top3.values.tolist(),
})
display(top3_comparison)

## 6. Hard-case listing

The rubric permits a listing that is a genuine outlier on the top-ranked feature. The procedure below is reproducible:

1. take the embedded method's top-ranked feature;
2. if it is numeric/non-binary, use the standard 1.5×IQR rule;
3. choose the outlier farthest from the feature median;
4. if no IQR outlier exists, choose the largest robust deviation and flag that fallback.

Actual values for all model features are saved for the selected listing.

In [ ]:
top_feature = embedded_top3.index[0]

if top_feature in DISCRETE_FEATURES:
    # A binary feature cannot be a magnitude outlier. Fall back to the
    # highest-ranked non-binary embedded feature.
    non_binary_ranked = [f for f in embedded_scores.index if f not in DISCRETE_FEATURES]
    if not non_binary_ranked:
        raise ValueError("No non-binary feature available for hard-case outlier selection.")
    hard_feature = non_binary_ranked[0]
else:
    hard_feature = top_feature

x = pd.to_numeric(df[hard_feature], errors="coerce")
q1, q3 = x.quantile([0.25, 0.75])
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
median = x.median()

outlier_mask = x.lt(lower) | x.gt(upper)
candidate_idx = x.loc[outlier_mask].dropna().index
used_iqr_outlier = len(candidate_idx) > 0

if used_iqr_outlier:
    hard_idx = (x.loc[candidate_idx] - median).abs().idxmax()
else:
    valid_idx = x.dropna().index
    hard_idx = (x.loc[valid_idx] - median).abs().idxmax()

hard_cols = [ID_COL, "split", TARGET, "price_clean"] + FEATURES
hard_case = df.loc[[hard_idx], hard_cols].copy()
hard_case.insert(1, "hard_case_feature", hard_feature)
hard_case.insert(2, "used_1_5_iqr_outlier_rule", used_iqr_outlier)
hard_case.insert(3, "feature_q1", q1)
hard_case.insert(4, "feature_q3", q3)
hard_case.insert(5, "feature_iqr", iqr)
hard_case.insert(6, "lower_bound", lower)
hard_case.insert(7, "upper_bound", upper)
hard_case.insert(8, "feature_median", median)

display(hard_case.T)

## 7. Concrete opposite-ranking scenario

This table identifies the feature with the largest rank disagreement. It provides the actual group-specific feature needed to explain how a filter method can rank a feature highly while the embedded tree ranks it lower (or vice versa).

In [ ]:
largest_disagreement = rank_table.sort_values(
    ["rank_difference", "filter_rank"],
    ascending=[False, True],
).iloc[[0]]

display(largest_disagreement)

## 8. Save reproducible feature-selection outputs

In [ ]:
TABLE_OUT = REPO_ROOT / "output" / "tables"
TABLE_OUT.mkdir(parents=True, exist_ok=True)

rank_table.to_csv(TABLE_OUT / "feature_selection_rankings.csv", index=False)
top3_comparison.to_csv(TABLE_OUT / "feature_selection_top3.csv", index=False)
hard_case.to_csv(TABLE_OUT / "feature_selection_hard_case.csv", index=False)
largest_disagreement.to_csv(
    TABLE_OUT / "feature_selection_largest_rank_disagreement.csv",
    index=False,
)

print("Saved:")
for name in [
    "feature_selection_rankings.csv",
    "feature_selection_top3.csv",
    "feature_selection_hard_case.csv",
    "feature_selection_largest_rank_disagreement.csv",
]:
    print(" -", TABLE_OUT / name)

## 9. Evidence checklist

Before the group-written report is finalised, verify:
- embedded top 3 + scores are quoted correctly;
- filter top 3 + scores are quoted correctly;
- disagreement is explained using the actual rank table;
- the hard-case listing ID and relevant actual values are quoted;
- the opposite-ranking scenario names a real feature from this group's feature set;
- feature-selection limitations are specific to the observed results, not generic.